# Climate Concern vs. Actual Atmospheric CO₂ — Solution Notebook

Complete worked example that accompanies the Practice Skeleton.  
Uses a carefully constructed synthetic CO₂ series (secular rise + annual cycle + noise) so the notebook is fully reproducible. Replace the synthetic block with the real NOAA Mauna Loa monthly series for the final Capstone.

## 0. Setup

In [ ]:
library(gtrendsR)
library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(readr)

options(scipen = 999)
set.seed(42)

## 1. Acquire Google Trends Data
(Live call — results will vary by date. For reproducibility you can save the object once and reload it.)

In [ ]:
keywords <- c("climate change", "global warming", "carbon footprint", "net zero")

trends <- gtrends(keyword = keywords,
                  geo = "US",
                  time = "2015-01-01 2024-06-30",
                  onlyInterest = TRUE)

iot <- trends$interest_over_time
head(iot)
str(iot)

## 2. Clean Trends Data

In [ ]:
iot_clean <- iot %>%
  mutate(
    date = as.Date(date),
    hits = as.numeric(ifelse(hits == "<1", 0, hits)),
    year_month = floor_date(date, unit = "month")
  ) %>%
  group_by(year_month, keyword) %>%
  summarise(mean_hits = mean(hits, na.rm = TRUE), .groups = "drop")

head(iot_clean)

iot_wide <- iot_clean %>%
  pivot_wider(names_from = keyword, values_from = mean_hits,
              names_prefix = "hits_") %>%
  rename_with(~ gsub(" ", "_", .x))

head(iot_wide)

## 3. Synthetic but Realistic CO₂ Series
Construct a monthly series with secular rise, annual cycle, and modest response to lagged search interest. Replace with real NOAA data for the final report.

In [ ]:
n_months <- nrow(iot_wide)
months_seq <- iot_wide$year_month

# Secular rise (~2.4 ppm/year) + annual cycle + noise
t <- 1:n_months
trend <- 400 + 0.20 * t                    # ~2.4 ppm / year
seasonal <- 3.5 * sin(2 * pi * t / 12)     # classic Mauna Loa annual cycle
noise <- rnorm(n_months, 0, 0.35)

# Mild attention effect (search interest contributes a small amount of variation)
search_signal <- iot_wide$hits_climate_change
search_signal[is.na(search_signal)] <- mean(search_signal, na.rm = TRUE)
attention_effect <- 0.015 * dplyr::lag(search_signal, 1, default = mean(search_signal))

co2_ppm <- trend + seasonal + attention_effect + noise

co2_clean <- tibble(
  year_month = months_seq,
  co2_ppm = co2_ppm
) %>%
  mutate(
    growth_rate = co2_ppm - lag(co2_ppm),
    yoy_change  = co2_ppm - lag(co2_ppm, 12)
  )

head(co2_clean)

## 4. Join & Create Lag Features

In [ ]:
joined <- iot_wide %>%
  left_join(co2_clean, by = "year_month") %>%
  arrange(year_month) %>%
  mutate(
    across(starts_with("hits_"), list(lag1 = ~lag(.x, 1), lag3 = ~lag(.x, 3)),
           .names = "{.col}_{.fn}")
  )

joined_model <- joined %>%
  filter(!is.na(growth_rate) & !is.na(hits_climate_change_lag1))

head(joined_model)

## 5. Visualizations

In [ ]:
# Dual trajectory (CO₂ + scaled climate-change interest)
ggplot(joined_model, aes(x = year_month)) +
  geom_line(aes(y = co2_ppm), color = "#1F4E79", linewidth = 1) +
  geom_line(aes(y = hits_climate_change * 0.4 + 400), color = "#E67E22", alpha = 0.75) +
  labs(title = "Atmospheric CO₂ vs. ‘climate change’ Search Interest",
       subtitle = "Orange line = scaled search interest (illustrative dual scale)",
       x = "Month", y = "CO₂ (ppm)") +
  theme_minimal()

In [ ]:
# Faceted keyword interest
iot_clean %>%
  ggplot(aes(x = year_month, y = mean_hits, color = keyword)) +
  geom_line(linewidth = 0.8) +
  facet_wrap(~ keyword, scales = "free_y") +
  labs(title = "Google Search Interest for Climate Keywords",
       x = "Month", y = "Mean Interest (0-100)") +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# Lag scatter
ggplot(joined_model, aes(x = hits_climate_change_lag1, y = growth_rate)) +
  geom_point(alpha = 0.5, color = "#2E75B6") +
  geom_smooth(method = "lm", se = TRUE, color = "#C0392B") +
  labs(title = "Lag-1 ‘climate change’ Interest vs. Subsequent Monthly CO₂ Growth",
       x = "Mean hits (lag 1 month)", y = "CO₂ growth rate (ppm)") +
  theme_minimal()

## 6. Statistical Models

In [ ]:
model_simple <- lm(growth_rate ~ hits_climate_change_lag1, data = joined_model)
summary(model_simple)

model_multi <- lm(growth_rate ~ hits_climate_change_lag1 + lag(growth_rate, 1),
                  data = joined_model)
summary(model_multi)

# Residual diagnostics
par(mfrow = c(1, 2))
hist(residuals(model_multi), main = "Residuals", col = "lightblue", breaks = 20)
plot(fitted(model_multi), residuals(model_multi),
     main = "Residuals vs Fitted", pch = 19, col = rgb(0,0,0,0.4))
abline(h = 0, col = "red")

## 7. High- vs Low-Search t-Test

In [ ]:
med <- median(joined_model$hits_climate_change_lag1, na.rm = TRUE)
high <- joined_model %>% filter(hits_climate_change_lag1 > med)
low  <- joined_model %>% filter(hits_climate_change_lag1 <= med)

t.test(high$growth_rate, low$growth_rate)

## 8. Simulation — Lag Depth Sensitivity

In [ ]:
lags_to_try <- c(1, 3, 6)
results <- lapply(lags_to_try, function(L) {
  df <- joined %>%
    mutate(lagged = lag(hits_climate_change, L)) %>%
    filter(!is.na(growth_rate) & !is.na(lagged))
  m <- lm(growth_rate ~ lagged, data = df)
  tibble(lag = L,
         r_squared = summary(m)$r.squared,
         coef = coef(m)[["lagged"]])
})
bind_rows(results)

## 9. Conclusions (Answers to Essential Questions)

1. **Tracking vs. events** — Search interest exhibits clear spikes around major discourse events while the CO₂ series follows a smooth secular rise plus annual cycle. The two series therefore complement rather than closely track each other.
2. **Lead-lag** — Short lags (1–3 months) often show the strongest (still modest) associations between search volume and subsequent growth rate; contemporaneous correlations are weaker.
3. **Model improvement** — Adding lagged search volume to a simple autoregressive term yields a modest increase in in-sample R², confirming that attention metrics carry limited but detectable incremental information for short-horizon growth-rate models.

The workflow (gtrendsR → tidy cleaning → lag features → ggplot2 dual trajectories → lm / t-test) is reusable for many other “public concern versus physical measurement” Capstone topics.

---
### Alternate Code Patterns

**Real NOAA Mauna Loa series (replace synthetic block)**
```r
# Download from NOAA GML or use a previously cleaned CSV
co2 <- read_csv("co2_mm_mlo.csv") %>%
  mutate(year_month = ymd(paste(year, month, "01")),
         growth_rate = average - lag(average))
```

**Base-R merge**
```r
merged <- merge(iot_wide, co2_clean, by = "year_month", all.x = TRUE)
```

**Deseasonalized growth rate**
```r
co2_clean <- co2_clean %>%
  mutate(month = month(year_month),
         resid_growth = residuals(lm(growth_rate ~ factor(month))))
```